In [84]:
from ete3 import NCBITaxa, TreeStyle
import requests
import os
import pandas as pd
import time

def get_taxon_id_batch(scientific_names):
    """
    Fetches NCBI Taxonomy IDs for a batch of scientific names using a two-step E-utilities approach.
    """
    # Format the terms for NCBI search: "Name 1"[sciname] OR "Name 2"[sciname]
    search_terms = " OR ".join([f'"{name}"[sciname]' for name in scientific_names])
    
    # 1. ESEARCH: Get the list of IDs for all valid names
    search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    search_params = {
        "db": "taxonomy",
        "term": search_terms,
        "retmode": "json",
        "retmax": len(scientific_names) * 2 # Buffer in case of multiple hits
    }
    
    try:
        search_response = requests.get(search_url, params=search_params)
        search_response.raise_for_status()
        id_list = search_response.json().get("esearchresult", {}).get("idlist", [])
        
        if not id_list:
            return {name: {"error": "No matches found"} for name in scientific_names}

        # 2. ESUMMARY: Fetch details for those IDs to map them back to the names
        summary_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"
        summary_params = {
            "db": "taxonomy",
            "id": ",".join(id_list),
            "retmode": "json"
        }
        
        summary_response = requests.get(summary_url, params=summary_params)
        summary_response.raise_for_status()
        summary_data = summary_response.json().get("result", {})
        
        # Build a mapping dictionary: { "homo sapiens": 9606, ... }
        # (Using lowercase for safer matching)
        name_to_id_map = {}
        for uid in summary_data.get("uids", []):
            sci_name = summary_data[uid].get("scientificname", "").lower()
            tax_id = summary_data[uid].get("taxid")
            name_to_id_map[sci_name] = tax_id

        # Compile final results matching the exact input
        results = {}
        for name in scientific_names:
            lower_name = name.lower()
            if lower_name in name_to_id_map:
                results[name] = {"Scientific Name": name, "Taxon ID": name_to_id_map[lower_name]}
            else:
                results[name] = {"error": "Taxon ID not found. Check spelling."}
                
        return results

    except requests.exceptions.RequestException as e:
        return {"error": f"API request failed: {e}"}

def get_species_batch_uniprot(uniprot_ids):
    """
    Fetches species/organism information for a batch of UniProt IDs in a single API call.
    Accepts a list of UniProt IDs.
    """
    # Join the list of IDs into a single comma-separated string
    accessions_string = ",".join(uniprot_ids)
    
    # The batch accessions endpoint
    url = "https://rest.uniprot.org/uniprotkb/accessions"
    params = {
        "accessions": accessions_string
    }
    
    try:
        # Make a single GET request for the whole batch
        response = requests.get(url, params=params)
        response.raise_for_status()
        
        data = response.json()
        results = {}
        
        # The API returns a 'results' array containing data for each valid ID
        for entry in data.get('results', []):
            uniprot_id = entry.get('primaryAccession')
            organism = entry.get('organism', {})
            
            results[uniprot_id] = {
                'Scientific Name': organism.get('scientificName', 'N/A'),
                'Common Name': organism.get('commonName', 'N/A'),
                'Taxon ID': organism.get('taxonId', 'N/A')
            }
            
        # Check if any requested IDs were missing from the response
        for uid in uniprot_ids:
            if uid not in results:
                results[uid] = {"error": "Not found or invalid ID"}
                
        return results

    except requests.exceptions.RequestException as e:
        return {"error": f"An error occurred: {e}"}
    

def get_taxon_data_from_synonyms(search_terms, sleep_time=1):
    """
    Fetches NCBI Taxon IDs using synonyms or common names, maps them back to the 
    original search term, and retrieves the official scientific name.
    """
    results = {}
    term_to_id_map = {}
    
    session = requests.Session()
    search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    for term in search_terms:
        params = {
            "db": "taxonomy",
            "term": term,
            "retmode": "json"
        }
        try:
            response = session.get(search_url, params=params)
            response.raise_for_status()
            id_list = response.json().get("esearchresult", {}).get("idlist", [])
            
            if id_list:
                term_to_id_map[term] = id_list[0]
            else:
                results[term] = {"error": "No match found for this synonym."}
                
        except requests.exceptions.RequestException as e:
            results[term] = {"error": f"Search failed: {e}"}
            
        time.sleep(sleep_time) 

    if term_to_id_map:
        unique_ids = list(set(term_to_id_map.values()))
        summary_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"
        summary_params = {
            "db": "taxonomy",
            "id": ",".join(unique_ids),
            "retmode": "json"
        }
        
        try:
            summary_response = session.get(summary_url, params=summary_params)
            summary_response.raise_for_status()
            summary_data = summary_response.json().get("result", {})
            id_to_official_name = {}
            for uid in unique_ids:
                if uid in summary_data:
                    id_to_official_name[uid] = summary_data[uid].get("scientificname", "Unknown")
            
            for term, tax_id in term_to_id_map.items():
                results[term] = {
                    "Taxon ID": int(tax_id),
                    "Official Scientific Name": id_to_official_name.get(tax_id, "Unknown")
                }
                
        except requests.exceptions.RequestException as e:
            print(f"Failed to fetch official names: {e}")
            for term, tax_id in term_to_id_map.items():
                results[term] = {"Taxon ID": int(tax_id), "Official Scientific Name": "Fetch Failed"}

    return results
    
# Get directory of the script
script_dir = os.path.dirname(os.getcwd())
repo_name = "WP2_SweetProteinDiscovery"
repo_dir = script_dir.split(repo_name)[0] + repo_name

onekp_sra_df = pd.read_csv(f"{repo_dir}/1_viridiplantae_prot_db/1_oneKP_capstone_2019/onekp_SRA.csv")
all_hits = f"{repo_dir}/6_structural_analyses/1_tm_scores/all_tm_scores.tsv"

In [99]:
# Read the TM-scores into a DataFrame
tm_scores_df = pd.read_csv(all_hits, sep='\t')

for index, row in tm_scores_df.iterrows():
    if row["source"] == "OneKP":
        accession = row["target"].replace("OneKP_scaffold-", "")
        accession = accession.split("-")[0]        
    elif row["source"] == "UniProt":
        accession = row["target"].split("|")[1]
    else:
        print(f"Unknown source {row['source']} for target {row['target']}. Skipping.")
        continue

    # Set accession 
    tm_scores_df.at[index, "accession"] = accession

# Get list of onekp ids from tm_scores
onekp_ids = set(tm_scores_df[tm_scores_df["source"] == "OneKP"]["accession"].tolist())

onekp_ncbi_ids_df = pd.read_excel(f"{repo_dir}/1_viridiplantae_prot_db/1_oneKP_capstone_2019/NCBI-ENA-Sequence-Identifiers.xlsx")
onekp_ncbi_ids_df = onekp_ncbi_ids_df[onekp_ncbi_ids_df["Sample Title"].isin(onekp_ids) | onekp_ncbi_ids_df["Should Be Combined"].isin(onekp_sra_df["OneKP_ID"].tolist())]
onekp_ncbi_ids_df = onekp_ncbi_ids_df[["Sample Title", "Tax_ID", "Organism"]]

missing_ids = [id for id in onekp_ids if id not in onekp_ncbi_ids_df["Sample Title"].values]
len(missing_ids)

205

In [101]:
for index, row in tm_scores_df[tm_scores_df["accession"].isin(missing_ids)].iterrows():
    print(f"Missing ID: {row['accession']} from target {row['target']}")

Missing ID: BMIF from target OneKP_scaffold-BMIF-2000309-Odontosororia_sp.
Missing ID: BMIF from target OneKP_scaffold-BMIF-2000308-Odontosororia_sp.
Missing ID: ZRAV from target OneKP_scaffold-ZRAV-2003287-ZRAV-Polypodium_hesperium-2_samples_combined
Missing ID: ZRAV from target OneKP_scaffold-ZRAV-2003289-ZRAV-Polypodium_hesperium-2_samples_combined
Missing ID: BMIF from target OneKP_scaffold-BMIF-2002223-Odontosororia_sp.
Missing ID: HDSY from target OneKP_scaffold-HDSY-2127503-HDSY-Aerva_persica-2_samples_combined
Missing ID: YKQR from target OneKP_scaffold-YKQR-2067758-YKQR-Hamamelis_virginiana-2_samples_combined
Missing ID: AAXJ from target OneKP_scaffold-AAXJ-2052672-AAXJ-Atriplex_prostrata-2_samples_combined
Missing ID: PIYM from target OneKP_scaffold-PIYM-2079224-PIYM-Galax_urceolata-2_samples_combined
Missing ID: NWQC from target OneKP_scaffold-NWQC-2002162-Plagiochila_asplenioides
Missing ID: AAXJ from target OneKP_scaffold-AAXJ-2052105-AAXJ-Atriplex_prostrata-2_samples_comb

In [86]:
# Get taxonomic information for UniProt hits
uniprot_ids = set(tm_scores_df[tm_scores_df["source"] == "UniProt"]["accession"].tolist())

# Split uniprot_ids into batches of 100
batch_size = 1000
uniprot_species_mapping = {}

for i in range(0, len(uniprot_ids), batch_size):
    print(f"Processing batch {i // batch_size + 1} of {((len(uniprot_ids) - 1) // batch_size) + 1}...")
    batch_ids = list(uniprot_ids)[i:i + batch_size]
    batch_results = get_species_batch_uniprot(batch_ids)
    uniprot_species_mapping.update(batch_results)


Processing batch 1 of 27...
Processing batch 2 of 27...
Processing batch 3 of 27...
Processing batch 4 of 27...
Processing batch 5 of 27...
Processing batch 6 of 27...
Processing batch 7 of 27...
Processing batch 8 of 27...
Processing batch 9 of 27...
Processing batch 10 of 27...
Processing batch 11 of 27...
Processing batch 12 of 27...
Processing batch 13 of 27...
Processing batch 14 of 27...
Processing batch 15 of 27...
Processing batch 16 of 27...
Processing batch 17 of 27...
Processing batch 18 of 27...
Processing batch 19 of 27...
Processing batch 20 of 27...
Processing batch 21 of 27...
Processing batch 22 of 27...
Processing batch 23 of 27...
Processing batch 24 of 27...
Processing batch 25 of 27...
Processing batch 26 of 27...
Processing batch 27 of 27...


In [87]:
# Split onekp_ids into batches of 50
scientific_names = set(onekp_sra_df[onekp_sra_df["OneKP_ID"].isin(missing_ids)]["Species"].tolist())
batch_size = 50
onekp_species_mapping = {}

for i in range(0, len(scientific_names), batch_size):
    print(f"Processing batch {i // batch_size + 1} of {((len(scientific_names) - 1) // batch_size) + 1}...")
    batch_ids = list(scientific_names)[i:i + batch_size]
    batch_results = get_taxon_id_batch(batch_ids)
    onekp_species_mapping.update(batch_results)
    time.sleep(1)  

# In case id mapping fails, try synoynm search
fails = []
for species in onekp_species_mapping:
    if "error" in onekp_species_mapping[species]:
        fails.append(species)
correct_id_mapping = get_taxon_data_from_synonyms(fails)


In [59]:
# Drop keys with errors and update with correct mapping
for species in fails:
    if species in onekp_species_mapping:
        del onekp_species_mapping[species]
    if species in correct_id_mapping and "error" not in correct_id_mapping[species]:
        onekp_species_mapping[species] = correct_id_mapping[species]
        del correct_id_mapping[species]

# Print any remaining errors
for species in correct_id_mapping:
    if "error" in correct_id_mapping[species]:
        print(f"Failed to resolve {species}: {correct_id_mapping[species]['error']}")

Failed to resolve Chloromonas rosae v. psychroph: No match found for this synonym.
Failed to resolve Lycopersicon cheesmanii: No match found for this synonym.


In [78]:
onekp_sra_df = onekp_sra_df[onekp_sra_df["OneKP_ID"].isin(missing_ids)]
for index, row in onekp_sra_df.iterrows():
    species = row["Species"]
    if species in onekp_species_mapping:
        tax_info = onekp_species_mapping[species]
        if "Taxon ID" in tax_info:
            onekp_sra_df.at[index, "Taxon ID"] = tax_info["Taxon ID"]
        if "Official Scientific Name" in tax_info:
            onekp_sra_df.at[index, "Official Scientific Name"] = tax_info["Official Scientific Name"] 

In [107]:
for index, row in tm_scores_df.iterrows():
    if row["source"] == "OneKP":
        try:
            species = onekp_sra_df[onekp_sra_df["OneKP_ID"] == row["accession"]]["Species"].values[0]
            taxon_id = onekp_sra_df[onekp_sra_df["OneKP_ID"] == row["accession"]]["Taxon ID"].values[0]
            tm_scores_df.at[index, "species"] = species
            tm_scores_df.at[index, "taxon_id"] = taxon_id
        except Exception as e:
            tm_scores_df.at[index, "species"] = "Unknown"
            tm_scores_df.at[index, "taxon_id"] = "Unknown"
    elif row["source"] == "UniProt":
        species_info = uniprot_species_mapping.get(row["accession"], {})
        tm_scores_df.at[index, "species"] = species_info.get("Scientific Name", "Unknown")
        tm_scores_df.at[index, "taxon_id"] = species_info.get("Taxon ID", "Unknown")
    else:
        tm_scores_df.at[index, "species"] = "Unknown"
        tm_scores_df.at[index, "taxon_id"] = "Unknown"

In [ ]:
uniprot_df = tm_scores_df[tm_scores_df["source"] == "UniProt"]
uniprot_df["taxon_id"].value_counts()

taxon_id
4565.0      819
3708.0      541
240449.0    441
183260.0    429
4113.0      408
           ... 
58860.0       1
178609.0      1
3435.0        1
269174.0      1
3046.0        1
Name: count, Length: 673, dtype: int64

In [ ]:
uniprot_df["query"].value_counts()

query
Thaumatin-I_1LR2           14662
Thaumatin-II_3AOK          14653
Miraculin                   6965
MNEI_1IV7                   3960
Honey_truffle_sweetener      799
Brazzein_4HE7                195
Name: count, dtype: int64

In [ ]:
for species in uniprot_df["species"].unique():
    count = uniprot_df[uniprot_df["species"] == species]["query"].nunique()
    proteins = uniprot_df[uniprot_df["species"] == species]["query"].unique()
    print(f"{species}: {count} unique queries, proteins: {proteins}")
#uniprot_df[uniprot_df["species"] == "Saponaria officinalis"]["query"].nunique()

Thaumatococcus daniellii: 2 unique queries, proteins: ['Thaumatin-II_3AOK' 'Thaumatin-I_1LR2']
Musa troglodytarum: 5 unique queries, proteins: ['Thaumatin-II_3AOK' 'Thaumatin-I_1LR2' 'MNEI_1IV7' 'Miraculin'
 'Honey_truffle_sweetener']
Acacia crassicarpa: 4 unique queries, proteins: ['Thaumatin-II_3AOK' 'Thaumatin-I_1LR2' 'MNEI_1IV7' 'Miraculin']
Musa acuminata subsp. malaccensis: 5 unique queries, proteins: ['Thaumatin-II_3AOK' 'Thaumatin-I_1LR2' 'MNEI_1IV7' 'Miraculin'
 'Honey_truffle_sweetener']
Colocasia esculenta: 4 unique queries, proteins: ['Thaumatin-II_3AOK' 'Thaumatin-I_1LR2' 'MNEI_1IV7' 'Miraculin']
Rhizophora mucronata: 4 unique queries, proteins: ['Thaumatin-II_3AOK' 'Thaumatin-I_1LR2' 'MNEI_1IV7' 'Miraculin']
Dioscorea cayennensis subsp. rotundata: 4 unique queries, proteins: ['Thaumatin-II_3AOK' 'Thaumatin-I_1LR2' 'MNEI_1IV7' 'Miraculin']
Tetracentron sinense: 4 unique queries, proteins: ['Thaumatin-II_3AOK' 'Thaumatin-I_1LR2' 'MNEI_1IV7' 'Miraculin']
Eucalyptus grandis: 

: 

In [122]:
# Initialize the NCBI Taxonomy database
ncbi = NCBITaxa()

# Step 1: Provide your list of plant species
# Replace these with the actual scientific names of your plants
species_list = uniprot_df[uniprot_df["query"] == "Brazzein_4HE7"]["species"].unique().tolist()

# Step 2: Translate species names to NCBI Taxonomic IDs (TaxIDs)
name2taxid = ncbi.get_name_translator(species_list)

# Flatten the dictionary to get a simple list of TaxIDs
taxids = [taxid[0] for taxid in name2taxid.values()]

# Step 3: Generate the taxonomic tree topology
tree = ncbi.get_topology(taxids)

# Step 4: Visualize the tree

# Option A: Print a quick ASCII tree directly to your console
print("ASCII Representation of the Species Tree:")
print(tree.get_ascii(attributes=["sci_name"]))

# Option B: Render the tree as an image file (e.g., PNG or PDF)
# We use a custom TreeStyle to show the scientific names at the tips
ts = TreeStyle()
ts.show_leaf_name = True
ts.show_branch_length = False
ts.show_branch_support = False

# Save to file
tree.render("plant_diversity_tree.png", tree_style=ts)
print("\nTree successfully rendered and saved as 'plant_diversity_tree.png'.")

ASCII Representation of the Species Tree:

                                                                  /Brassica oleracea-Brassica oleracea var. oleracea
                                                                 |
                                                                 |             /-Brassica rapa subsp. trilocularis
                                                                 |-Brassica rapa
                                                                 |             \-Brassica rapa subsp. oleifera
                                                          /Brassica
                                                         |       |--Brassica cretica
                                                         |       |
                                                         |       |--Brassica carinata
                                                         |       |
                                                /Brassiceae       \-Brassica napus
          